STEP 1: Install Required Packages

In [57]:
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install google-genai
%pip install langextract

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP 2: Import All Libraries

In [ ]:
# Numpy and Pandas for data manipulation and analysis
import numpy as np
import pandas as pd 

# Sklearn modules for model training, testing, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# os module to access environment variables for API key
import os

# google-genai for Gemini API access
from google.genai import Client

STEP 3: Load Dataset

In [ ]:
# Load the dataset from the CSV file
df = pd.read_csv('indian_finance_ml_dataset_balanced_final (1).csv')

STEP 4: Train ML Model (Support Vector Classifier (SVC)) For Predicting Financial State Category

In [ ]:
# Select specific columns for features (X) and the multiclass target (y)
X = df[['Monthly Income (INR)', 'Cost of Living Expenditure (INR)', 
         'Other Important Investments (INR)', 'Consumerist Expenditure (INR)', 
         'Crisis Shock Expenditure (INR)', 'Total Monthly Expenditure (INR)', 
         'Debt Status']]
y = df['Financial State Category']  

# Categorize features into numeric and categorical for appropriate preprocessing
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = ['Debt Status'] 

# Handle missing numerical values with median imputation and normalize them via scaling
numeric_transformer_X = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Use LabelEncoder for the categorical feature and target variable to convert text labels into integer labels
categorical_transformer_X = LabelEncoder() 
categorical_transformer_y = LabelEncoder()  # For the target variable, we will use LabelEncoder instead of OneHotEncoder               

# Apply the transformations to the feature set (X) and target variable (y)
numeric_transformer_X.fit(X[numeric_features])
X[numeric_features] = numeric_transformer_X.transform(X[numeric_features])

categorical_transformer_X.fit(X[categorical_features].values.ravel())
X[categorical_features] = categorical_transformer_X.transform(X[categorical_features].values.ravel()).reshape(-1, 1)

y_transformed = categorical_transformer_y.fit_transform(y)

# 3. Partition data into training arrays (80%) and testing validation arrays (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

# 4. Initialize and fit a Support Vector Classifier (SVC) using a linear kernel
# class_weight='balanced' automatically counteracts potential label imbalances
model = SVC(kernel='linear', C=1.0, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# 5. Generate target predictions on the unseen testing feature split
y_pred = model.predict(X_test)

# 6. Evaluate accuracy metrics and output performance reports
accuracy = accuracy_score(y_test, y_pred)
print(f"\nSVM Model Accuracy: {accuracy:.4f}\n")

# Use decoded class labels inside the evaluation reports for readable text results
print("SVM Classification Report:")
print(classification_report(y_test, y_pred, target_names=categorical_transformer_y.classes_.astype(str)))   

print("SVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))




SVM Model Accuracy: 0.8750

SVM Classification Report:
                                    precision    recall  f1-score   support

                    Deficit Living       0.84      1.00      0.91        51
High-Rate Saver (>=25% in savings)       1.00      0.98      0.99        58
  Low-Rate Saver (<25% in savings)       1.00      0.55      0.71        42
               Zero-Balance Living       0.75      0.90      0.81        49

                          accuracy                           0.88       200
                         macro avg       0.90      0.86      0.86       200
                      weighted avg       0.90      0.88      0.87       200

SVM Confusion Matrix:
[[51  0  0  0]
 [ 1 57  0  0]
 [ 4  0 23 15]
 [ 5  0  0 44]]


STEP 5:  Train ML Model (Support Vector Classifier (SVC)) For Predicting Current Monthly Income Sufficiency & Worth of Current Expenditure

In [ ]:
# Separate independent variables (features) and the dependent variable (target)
X2 = df[['Monthly Income (INR)', 'Cost of Living Expenditure (INR)', 'Other Important Investments (INR)', 'Consumerist Expenditure (INR)', 'Crisis Shock Expenditure (INR)', 'Total Monthly Expenditure (INR)', 'Debt Status','Financial State Category']]  # Features matrix
y1 = df['Current Monthly Income Enough for Next Few Months']  
y2 = df['Current Expenditure Worth It']

# Categorize features into numeric and categorical for appropriate preprocessing
numeric_features_X2 = X2.select_dtypes(include=['int64', 'float64']).columns
categorical_features_X2 = X2.select_dtypes(include=['object', 'category']).columns

# Handle missing numerical values with median imputation and normalize them via scaling
numeric_transformer_X2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Use LabelEncoder for the categorical features and target variables to convert text labels into integer labels
categorical_transformer_X2_debt= LabelEncoder()
catergorical_transformer_X2_financial_state = LabelEncoder()

# Apply the transformations to the feature set (X2) and target variables (y1, y2)
numeric_transformer_X2.fit(X2[numeric_features_X2])
X2[numeric_features_X2] = numeric_transformer_X2.transform(X2[numeric_features_X2])

categorical_transformer_X2_debt.fit(X2['Debt Status'].values.ravel())
X2['Debt Status'] = categorical_transformer_X2_debt.transform(X2['Debt Status'].values.ravel()).reshape(-1, 1)

catergorical_transformer_X2_financial_state.fit(X2['Financial State Category'].values.ravel())
X2['Financial State Category'] = catergorical_transformer_X2_financial_state.transform(X2['Financial State Category'].values.ravel()).reshape(-1, 1)

le1 = LabelEncoder()
le2 = LabelEncoder()
y1_transformed = le1.fit_transform(y1)
y2_transformed = le2.fit_transform(y2)

# 3. Split the data into training (80%) and testing (20%) sets
X_train1, X_test1, y_train1, y_test1 = train_test_split(X2, y1_transformed, test_size=0.2, random_state=42)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2_transformed, test_size=0.2, random_state=42)

# 5. Initialize and fit Linear SVM model
model1 = SVC(kernel='linear', C=1.0, random_state=42, class_weight='balanced')
model2 = SVC(kernel='linear', C=1.0, random_state=42, class_weight='balanced')
model1.fit(X_train1, y_train1)
model2.fit(X_train2, y_train2)

# 6. Make predictions on the test dataset
y_pred1 = model1.predict(X_test1)
y_pred2 = model2.predict(X_test2)

# 7. Evaluate model performance
accuracy1 = accuracy_score(y_test1, y_pred1)
accuracy2 = accuracy_score(y_test2, y_pred2)

print(f"\nSVM Model 1 Accuracy: {accuracy1:.4f}\n")
print("SVM Model 1 Classification Report:")
print(classification_report(y_test1, y_pred1, target_names=le1.classes_.astype(str))) # Added class names
print("SVM Model 1 Confusion Matrix:")
print(confusion_matrix(y_test1, y_pred1))

print(f"\nSVM Model 2 Accuracy: {accuracy2:.4f}\n")
print("SVM Model 2 Classification Report:")
print(classification_report(y_test2, y_pred2, target_names=le2.classes_.astype(str))) # Added class names
print("SVM Model 2 Confusion Matrix:")
print(confusion_matrix(y_test2, y_pred2))



SVM Model 1 Accuracy: 0.9650

SVM Model 1 Classification Report:
              precision    recall  f1-score   support

          No       0.94      1.00      0.97       117
         Yes       1.00      0.92      0.96        83

    accuracy                           0.96       200
   macro avg       0.97      0.96      0.96       200
weighted avg       0.97      0.96      0.96       200

SVM Model 1 Confusion Matrix:
[[117   0]
 [  7  76]]

SVM Model 2 Accuracy: 0.9350

SVM Model 2 Classification Report:
              precision    recall  f1-score   support

          No       0.97      0.92      0.95       126
         Yes       0.88      0.96      0.92        74

    accuracy                           0.94       200
   macro avg       0.93      0.94      0.93       200
weighted avg       0.94      0.94      0.94       200

SVM Model 2 Confusion Matrix:
[[116  10]
 [  3  71]]


C:\Users\medhy\AppData\Local\Temp\ipykernel_30776\2721683062.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_X2 = X2.select_dtypes(include=['object', 'category']).columns


STEP 6:  Train ML Model (Random Forest Regressor) For Predicting Suggested Monthly Budget

In [ ]:
# Performing feature engineering to create new features that may help the model learn better patterns
# What percentage of their money is left over?
df['Savings_Margin_Ratio'] = (df['Monthly Income (INR)'] - df['Total Monthly Expenditure (INR)']) / df['Monthly Income (INR)']
# What percentage of total outflow goes to survival?
df['Essential_Cost_Ratio'] = df['Cost of Living Expenditure (INR)'] / df['Total Monthly Expenditure (INR)']
# Behavioral allocation
df['Current_Investment_Allocation_Rate'] = df['Other Important Investments (INR)'] / df['Monthly Income (INR)']
df['Current_Crisis_Allocation_Rate'] = df['Crisis Shock Expenditure (INR)'] / df['Monthly Income (INR)']

# Updating feature list to use the rates, NOT the raw Rupee values
feature_names = [
    'Monthly Income (INR)', 'Cost of Living Expenditure (INR)', 'Other Important Investments (INR)', 'Consumerist Expenditure (INR)',
    'Crisis Shock Expenditure (INR)', 'Total Monthly Expenditure (INR)',
    'Savings_Margin_Ratio', 'Essential_Cost_Ratio',
    'Current_Investment_Allocation_Rate', 'Current_Crisis_Allocation_Rate' # Relies on history safely!
]

# Separate independent variables (features) and the dependent variable (target)
X3 = df[feature_names]  
y3 = df['Suggested Budget - Cost of Living']  
y4 = df['Suggested Budget - Other Important Investments']  
y5 = df['Suggested Budget - Consumerist Expenditure']  
y6 = df['Suggested Budget - Crisis Shocks']  

# Handle missing values and scale numeric data
numeric_transformer_X3 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Apply the transformations to the feature set (X3)
X3_transformed = numeric_transformer_X3.fit_transform(X3)

# Split the data into training (80%) and testing (20%) sets
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3_transformed, y3, test_size=0.2, random_state=42)
X_train4, X_test4, y_train4, y_test4 = train_test_split(X3_transformed, y4, test_size=0.2, random_state=42)
X_train5, X_test5, y_train5, y_test5 = train_test_split(X3_transformed, y5, test_size=0.2, random_state=42)
X_train6, X_test6, y_train6, y_test6 = train_test_split(X3_transformed, y6, test_size=0.2, random_state=42)

# Initialize Random Forest Regressor models 
# random_state ensures reproducibility, n_estimators sets the number of decision trees
model3 = RandomForestRegressor(n_estimators=100, random_state=42)
model4 = RandomForestRegressor(n_estimators=100, random_state=42)
model5 = RandomForestRegressor(n_estimators=100, random_state=42)
model6 = RandomForestRegressor(n_estimators=100, random_state=42)

# Fit the Random Forest Regressors
model3.fit(X_train3, y_train3)
model4.fit(X_train4, y_train4)
model5.fit(X_train5, y_train5)
model6.fit(X_train6, y_train6)

# 6. Make predictions on the test dataset
y_pred3 = model3.predict(X_test3)
y_pred4 = model4.predict(X_test4)
y_pred5 = model5.predict(X_test5)
y_pred6 = model6.predict(X_test6)

# 7. Evaluate model performance metrics
mae3 = mean_absolute_error(y_test3, y_pred3)
mse3 = mean_squared_error(y_test3, y_pred3)
rmse3 = np.sqrt(mse3)  
r2_3 = r2_score(y_test3, y_pred3)

mae4 = mean_absolute_error(y_test4, y_pred4)
mse4 = mean_squared_error(y_test4, y_pred4)
rmse4 = np.sqrt(mse4)  
r2_4 = r2_score(y_test4, y_pred4)

mae5 = mean_absolute_error(y_test5, y_pred5)
mse5 = mean_squared_error(y_test5, y_pred5)
rmse5 = np.sqrt(mse5)  
r2_5 = r2_score(y_test5, y_pred5)

mae6 = mean_absolute_error(y_test6, y_pred6)
mse6 = mean_squared_error(y_test6, y_pred6)
rmse6 = np.sqrt(mse6)  
r2_6 = r2_score(y_test6, y_pred6)

# Print evaluation summaries
print("--- Model 3 (Random Forest) Performance Metrics ---")
print(f"Mean Absolute Error: {mae3:.2f}")
print(f"Mean Squared Error (MSE)       : {mse3:.2f}")
print(f"Root Mean Squared Error (RMSE) : {rmse3:.2f}")
print(f"R-squared Score (R2)           : {r2_3:.4f}")

print("\n--- Model 4 (Random Forest) Performance Metrics ---")
print(f"Mean Absolute Error: {mae4:.2f}")
print(f"Mean Squared Error (MSE)       : {mse4:.2f}")
print(f"Root Mean Squared Error (RMSE) : {rmse4:.2f}")
print(f"R-squared Score (R2)           : {r2_4:.4f}")

print("\n--- Model 5 (Random Forest) Performance Metrics ---")
print(f"Mean Absolute Error: {mae5:.2f}")
print(f"Mean Squared Error (MSE)       : {mse5:.2f}")
print(f"Root Mean Squared Error (RMSE) : {rmse5:.2f}")
print(f"R-squared Score (R2)           : {r2_5:.4f}")

print("\n--- Model 6 (Random Forest) Performance Metrics ---")
print(f"Mean Absolute Error: {mae6:.2f}")
print(f"Mean Squared Error (MSE)       : {mse6:.2f}")
print(f"Root Mean Squared Error (RMSE) : {rmse6:.2f}")
print(f"R-squared Score (R2)           : {r2_6:.4f}")

models_info = [
    ("Model 3: Cost of Living Budget", model3),
    ("Model 4: Other Important Investments Budget", model4),
    ("Model 5: Consumerist Expenditure Budget", model5),
    ("Model 6: Crisis Shocks Budget", model6)
]

print("--- RANDOM FOREST FEATURE IMPORTANCE ANALYSIS ---\n")

for name, model in models_info:
    print(f"=== {name} ===")
    
    # Extract structural weights (importance scores add up to 1.0)
    importances = model.feature_importances_
    
    # Create a clean sorted DataFrame for readability
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance Share': importances
    }).sort_values(by='Importance Share', ascending=False)
    
    # Print out formatted shares
    for idx, row in importance_df.iterrows():
        print(f"  {row['Feature']:<35} : {row['Importance Share']*100:>6.2f}%")
    print("-" * 50)



--- Model 3 (Random Forest) Performance Metrics ---
Mean Absolute Error: 0.01
Mean Squared Error (MSE)       : 0.00
Root Mean Squared Error (RMSE) : 0.01
R-squared Score (R2)           : 0.9922

--- Model 4 (Random Forest) Performance Metrics ---
Mean Absolute Error: 0.00
Mean Squared Error (MSE)       : 0.00
Root Mean Squared Error (RMSE) : 0.00
R-squared Score (R2)           : 0.9917

--- Model 5 (Random Forest) Performance Metrics ---
Mean Absolute Error: 0.01
Mean Squared Error (MSE)       : 0.00
Root Mean Squared Error (RMSE) : 0.01
R-squared Score (R2)           : 0.7311

--- Model 6 (Random Forest) Performance Metrics ---
Mean Absolute Error: 0.00
Mean Squared Error (MSE)       : 0.00
Root Mean Squared Error (RMSE) : 0.01
R-squared Score (R2)           : 0.9299
--- RANDOM FOREST FEATURE IMPORTANCE ANALYSIS ---

=== Model 3: Cost of Living Budget ===
  Savings_Margin_Ratio                :  80.87%
  Monthly Income (INR)                :   9.58%
  Essential_Cost_Ratio             

STEP 7: Input User Financial Query and Extract Information From It Using Langextract

In [ ]:
import textwrap
import langextract as lx

# Define the extraction rules and desired schema
prompt = textwrap.dedent("""
    Extract financial information. 
    Identify if following information exists in text and extract info that exists:'Monthly Income (INR)', 'Cost of Living Expenditure (INR)', 'Other Important Investments (INR)', 'Consumerist Expenditure (INR)', 'Crisis Shock Expenditure (INR)', 'Total Monthly Expenditure (INR)', 'Debt Status'.
    Use exact text for extractions.
""")

# Provide a high-quality few-shot example to teach the model the layout
examples = [lx.data.ExampleData(
        text="Hey, I need some honest financial advice. I am currently working as a software engineer in Bengaluru, and my monthly in-hand salary is ₹1,40,000. Lately, I feel like my money is just vanishing. My fixed overheads are quite high—between my house rent, society maintenance, maid, electricity, and groceries, my core cost of living comes out to exactly ₹45,000 every month. On top of that, I am upskilling myself to transition into a management role, so I am paying a monthly EMI of ₹15,000 for an executive data science certification course from an IIM. I’ve also been overspending a bit on lifestyle choices; between dining out at pubs, weekend trips, and ordering from Swiggy/Zomato, my discretionary consumerist spending hits around ₹25,000 monthly. To make matters worse, my father had a minor medical emergency back home last month, so I’ve had to commit a fixed ₹15,000 monthly towards his ongoing healthcare and medicine costs for the foreseeable future. If you calculate it all, my total monthly expenditure has ballooned to ₹1,00,000, leaving me with very little savings. As for my overall debt status, I currently owe ₹3,50,000 on an active personal loan that I took out for relocation and furnishing last year. Looking at this breakdown, do you think my current level of expenditure is actually worth it, or am I jeopardizing my long-term financial health for short-term comfort?",
        extractions=[
            lx.data.Extraction(
                extraction_class="Monthly Income (INR)",
                extraction_text="140000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Cost of Living Expenditure (INR)",
                extraction_text="45000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Other Important Investments (INR)",
                extraction_text="15000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Consumerist Expenditure (INR)",
                extraction_text="25000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Crisis Shock Expenditure (INR)",
                extraction_text="15000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Total Monthly Expenditure (INR)",
                extraction_text="100000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Debt Status",
                extraction_text="In Debt",
                attributes={}
            )
        ]
    ),
    lx.data.ExampleData(
        text="Hey, I need some honest financial advice. I am currently working as a software engineer in Bengaluru, and my monthly in-hand salary is ₹1,40,000. Lately, I feel like my money is just vanishing. My fixed overheads are quite high—between my house rent, society maintenance, maid, electricity, and groceries, my core cost of living comes out to exactly ₹45,000 every month. On top of that, I am upskilling myself to transition into a management role, so I am paying a monthly EMI of ₹15,000 for an executive data science certification course from an IIM. I’ve also been overspending a bit on lifestyle choices; between dining out at pubs, weekend trips, and ordering from Swiggy/Zomato, my discretionary consumerist spending hits around ₹25,000 monthly. To make matters worse, my father had a minor medical emergency back home last month, so I’ve had to commit a fixed ₹15,000 monthly towards his ongoing healthcare and medicine costs for the foreseeable future. If you calculate it all, my total monthly expenditure has ballooned to ₹1,00,000, leaving me with very little savings. As for my overall debt status, I don't owe anything. Looking at this breakdown, do you think my current level of expenditure is actually worth it, or am I jeopardizing my long-term financial health for short-term comfort?",
        extractions=[
            lx.data.Extraction(
                extraction_class="Monthly Income (INR)",
                extraction_text="140000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Cost of Living Expenditure (INR)",
                extraction_text="45000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Other Important Investments (INR)",
                extraction_text="15000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Consumerist Expenditure (INR)",
                extraction_text="25000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Crisis Shock Expenditure (INR)",
                extraction_text="15000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Total Monthly Expenditure (INR)",
                extraction_text="100000",
                attributes={}
            ),
            lx.data.Extraction(
                extraction_class="Debt Status",
                extraction_text="Not in Debt",
                attributes={}
            )
        ]
    )]

classes = ['Monthly Income (INR)', 'Cost of Living Expenditure (INR)', 'Other Important Investments (INR)', 'Consumerist Expenditure (INR)', 'Crisis Shock Expenditure (INR)', 'Total Monthly Expenditure (INR)', 'Debt Status']

# Take user query input
query = input("What's your Financial Query?\nProvide maximum details related to your financial situation to get personalized advice on your income, expenses, and investment decisions.\n\n")
# Define user query as the unstructured raw target string that is to be evaluated for extractions
input_text = (
    query
)
# Execute the extraction pass
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash", # Balances cost, speed, and recall quality
) 

# Print the structured extraction results
X_validation = [int(extraction.extraction_text) for extraction in result.extractions if extraction.extraction_class != 'Debt Status']+[result.extractions[-1].extraction_text]
print("\n--- EXTRACTION RESULT ---")
print(X_validation)

LangExtract: model=gemini-2.5-flash, current=1,498 chars, processed=0 chars:  [00:05]


--- EXTRACTION RESULT ---
[185000, 52000, 35000, 30000, 20000, 137000, 'Not in Debt']


STEP 8: Predict Financial State Category Based on User Query Through Trained ML Model

In [ ]:
# Preprocess the extracted values and prepare them for prediction
X_validation_df=pd.DataFrame([X_validation], columns=classes)  # Create a DataFrame with the extracted values

numeric_features = classes[:-1]  # All except 'Debt Status'
categorical_features = ['Debt Status']             

numeric_transformer_X.fit(X_validation_df[numeric_features])
X_validation_df[numeric_features] = numeric_transformer_X.transform(X_validation_df[numeric_features])

categorical_transformer_X.fit(X_validation_df[categorical_features].values.ravel())
X_validation_df[categorical_features] = categorical_transformer_X.transform(X_validation_df[categorical_features].values.ravel()).reshape(-1, 1)

# Make a prediction using the trained SVM model
y_pred = model.predict(X_validation_df)  # Exclude the last element which is 'Debt Status' text
financial_state_category = categorical_transformer_y.inverse_transform(y_pred)[0]

# Print the predicted financial state category
print("\n--- PREDICTION RESULT ---")
print(f"Predicted Financial State Category: {financial_state_category}")





--- PREDICTION RESULT ---
Predicted Financial State Category: Low-Rate Saver (<25% in savings)


STEP 9: Predict Current Monthly Income Sufficiency & Worth Of Current Expenditure Based on User Query

In [ ]:
# Preprocess the extracted values and prepare them for prediction
X2_validation = X_validation+[y_pred[0]]
X2_validation_df = pd.DataFrame([X2_validation], columns=classes + ['Financial State Category'])  # Create a DataFrame with the extracted values

numeric_transformer_X2.fit(X2_validation_df[numeric_features_X2])
X2_validation_df[numeric_features_X2] = numeric_transformer_X2.transform(X2_validation_df[numeric_features_X2])

categorical_transformer_X2_debt.fit(X2_validation_df['Debt Status'].values.ravel())
X2_validation_df['Debt Status'] = categorical_transformer_X2_debt.transform(X2_validation_df['Debt Status'].values.ravel()).reshape(-1, 1)

catergorical_transformer_X2_financial_state.fit(X2_validation_df['Financial State Category'].values.ravel())
X2_validation_df['Financial State Category'] = catergorical_transformer_X2_financial_state.transform(X2_validation_df['Financial State Category'].values.ravel()).reshape(-1, 1)

# Make predictions using the trained SVM models for the two additional questions
y_pred1 = model1.predict(X2_validation_df)
y_pred2 = model2.predict(X2_validation_df)
current_monthly_income_enough = le1.inverse_transform(y_pred1)[0]
current_expenditure_worth_it = le2.inverse_transform(y_pred2)[0]

# Print the predictions for the additional questions
print("\n--- ADDITIONAL PREDICTIONS ---")
print(f"Is Current Monthly Income Enough for Next Few Months? {current_monthly_income_enough}")
print(f"Is Current Expenditure Worth It? {current_expenditure_worth_it}")


--- ADDITIONAL PREDICTIONS ---
Is Current Monthly Income Enough for Next Few Months? Yes
Is Current Expenditure Worth It? No


STEP 10: Predict Suggested Monthly Budget Based on Current Monthly Income and Expenditure

In [ ]:
# Preprocess the extracted values and prepare them for prediction for the budget suggestions
X3_validation = X_validation[:-1]

X3_validation_df = pd.DataFrame([X3_validation], columns=classes[:-1]) 

X3_validation_df['Savings_Margin_Ratio'] = (X3_validation_df['Monthly Income (INR)'] - X3_validation_df['Total Monthly Expenditure (INR)']) / X3_validation_df['Monthly Income (INR)']

# What percentage of total outflow goes to survival?
X3_validation_df['Essential_Cost_Ratio'] = X3_validation_df['Cost of Living Expenditure (INR)'] / X3_validation_df['Total Monthly Expenditure (INR)']

# Instead of raw Rupee values, teach the model the behavioral allocation
X3_validation_df['Current_Investment_Allocation_Rate'] = X3_validation_df['Other Important Investments (INR)'] / X3_validation_df['Monthly Income (INR)']
X3_validation_df['Current_Crisis_Allocation_Rate'] = X3_validation_df['Crisis Shock Expenditure (INR)'] / X3_validation_df['Monthly Income (INR)']



numeric_transformer_X3.fit(X3_validation_df)
X3_validation_df_transformed= numeric_transformer_X3.transform(X3_validation_df)

# Make predictions using the trained Random Forest Regressor models for the budget suggestions
y_pred3 = model3.predict(X3_validation_df_transformed)
y_pred4 = model4.predict(X3_validation_df_transformed)
y_pred5 = model5.predict(X3_validation_df_transformed)
y_pred6 = model6.predict(X3_validation_df_transformed)

# Print the suggested budgets for each expenditure category
print("\n--- SUGGESTED BUDGETS ---")
print(f"Suggested Budget for Cost of Living Expenditure (INR): {y_pred3[0]:.2f}")
print(f"Suggested Budget for Other Important Investments (INR): {y_pred4[0]:.2f}")
print(f"Suggested Budget for Consumerist Expenditure (INR): {y_pred5[0]:.2f}")
print(f"Suggested Budget for Crisis Shock Expenditure (INR): {y_pred6[0]:.2f}")


--- SUGGESTED BUDGETS ---
Suggested Budget for Cost of Living Expenditure (INR): 0.47
Suggested Budget for Other Important Investments (INR): 0.11
Suggested Budget for Consumerist Expenditure (INR): 0.09
Suggested Budget for Crisis Shock Expenditure (INR): 0.11


STEP 11: Generate Text Response Containing Financial Advice Through Gemini API 

In [22]:
# Initialize the client (automatically picks up GEMINI_API_KEY environment variable)
api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
        print("Error: GEMINI_API_KEY environment variable not set.")
        exit(1)
else:
        client = Client(api_key=api_key)

        df= pd.read_csv('indian_finance_ml_dataset_balanced_final (1).csv')

    # Construct a clear, structured prompt for the AI
        prompt = f"""
    You are Finan, an informative and supportive ai financial advisor.
    Give financial advice based on the following user query, extracted financial information and predicted financial information:
User Query: {query}
Monthly Income (INR): {X_validation[0]}
Cost of Living Expenditure (INR): {X_validation[1]}
Other Important Investments (INR): {X_validation[2]}
Consumerist Expenditure (INR): {X_validation[3]}
Crisis Shock Expenditure (INR): {X_validation[4]}
Total Monthly Expenditure (INR): {X_validation[5]}
Debt Status: {X_validation[6]}
Predicted Financial State Category: {financial_state_category}
Is Current Monthly Income Enough for Next Few Months? {current_monthly_income_enough}
Is Current Expenditure Worth It? {current_expenditure_worth_it}
Suggested Budget for Cost of Living Expenditure (INR): {(y_pred3[0]* X_validation[0]):.2f}
Suggested Budget for Other Important Investments (INR): {(y_pred4[0] * X_validation[0]):.2f}
Suggested Budget for Consumerist Expenditure (INR): {(y_pred5[0] * X_validation[0]):.2f}
Suggested Budget for Crisis Shock Expenditure (INR): {(y_pred6[0] * X_validation[0]):.2f}

Format for financial advice:
1. Greet user, tell your name as Finan, an informative and supportive ai financial advisor.
2. Acknowledge the user's financial situation with empathy and without judgment.
3. Provide a clear summary of user's financial situation by summarising all information extracted from user query.
4. Reveal user's predicted financial state category and explain what it means in simple terms.
5. Inform user whether their current monthly income is likely to be sufficient for the next few months and their current expenditure is likely to be worth it based on the model's prediction.
6. Display the suggested budgets values in INR for each expenditure category. 
7. Explain how suggested budgets can help improve user's financial health.
8. Based on the predicted financial state category, provide personalized financial advice on how to manage income based on indian economic context, how to optimize expenditure, and how to approach investments. Handle any other queries user has based on indian economic landscape. Avoid generic advice and focus on actionable steps user can take.
End the respone with a support note for user and wish them well for future and avoid using any technical terms in that note.
    """


    # Generate content using the fast, standard multimodal model
        response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
            )
        print(response.text)

Hello there! I'm Finan, your informative and supportive AI financial advisor. It sounds like you're carrying a significant load managing both your growing business and household responsibilities, and it's completely understandable to feel the weight of it all. It takes a lot of dedication to build a business, and looking for clarity on your financial path is a very wise step.

Let's break down your current financial situation:
You operate a digital marketing agency in Hyderabad, bringing in a variable but averaged monthly net income of **₹1,85,000**.
Your essential cost of living, covering rent, utilities, fuel, and basic groceries, is **₹52,000** per month.
You're heavily invested in business growth, with **₹35,000** monthly for premium software, cloud infrastructure, and professional networking.
Your discretionary lifestyle choices, including client dinners, trends, and shopping, amount to **₹30,000** monthly.
Recently, you've taken on an additional **₹20,000** monthly to support you